# 03 — Retraining policy comparison
Runs 32 week simulations (4 policies × 8 weeks). This may take a few minutes.

In [ ]:
import pandas as pd
from src.core import reset_artifacts, ensure_baseline, run_week
policies=[('never',0.10),('weekly',0.10),('drift_low',0.10),('drift_high',0.25)]
records=[]
for policy,threshold in policies:
    reset_artifacts()
    for w in range(1,9):
        r=run_week(w,policy,threshold)
        records.append(r)
results=pd.DataFrame(records)
results[['policy','week','status','retraining_triggered','promoted','escalation']]

In [ ]:
summary=[]
for policy,g in results.groupby('policy'):
    summary.append({'policy':policy,'failed_runs':int((g.status=='failed').sum()),
                    'retraining_attempts':int(g.get('retraining_triggered',pd.Series(False,index=g.index)).fillna(False).sum()),
                    'promotions':int(g.get('promoted',pd.Series(False,index=g.index)).fillna(False).sum()),
                    'retraining_cost_units':10*int(g.get('retraining_triggered',pd.Series(False,index=g.index)).fillna(False).sum())})
summary=pd.DataFrame(summary)
summary

In [ ]:
import matplotlib.pyplot as plt
for policy,g in results.groupby('policy'):
    g=g.sort_values('week')
    plt.plot(g['week'], g['promoted'].fillna(False).astype(int), marker='o', label=policy)
plt.xticks(range(1,9)); plt.xlabel('Week'); plt.ylabel('Promotion (1=yes)')
plt.title('Policy timeline: promotions by week'); plt.legend(); plt.tight_layout(); plt.show()